## Emotion Analysis

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from sklearn.feature_extraction.text import CountVectorizer
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from sklearn.model_selection import train_test_split
from keras.utils.np_utils import to_categorical
import re
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

In [ ]:
from sklearn import metrics, cross_validation

Data imported from: https://inclass.kaggle.com/c/sa-emotions

In [ ]:
data = pd.read_csv('../text_emotion.csv')
data.head()

Only keeping the necessary columns.

In [ ]:
# Keeping only the neccessary columns
data = data[['content','sentiment']]
data.head()

### Remove lessed labeled emotion for better prediction

In [ ]:
del_row=['empty','enthusiasm','boredom','anger']

In [ ]:
data=data[data['sentiment']!='empty']

In [ ]:
data=data[data['sentiment']!='enthusiasm']

In [ ]:
data=data[data['sentiment']!='boredom']

In [ ]:
data=data[data['sentiment']!='anger']

In [ ]:
data.shape

38125 data for 9 category prediction is very less encouraging 

In [ ]:
MAX_NB_WORDS = 20000

In [ ]:
MAX_SEQUENCE_LENGTH=30

In [ ]:
data['content'] = data['content'].apply(lambda x: x.lower())
data['content'] = data['content'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))

print(data['sentiment'].value_counts())

for idx,row in data.iterrows():
    row[0] = row[0].replace('rt',' ')

tokenizer = Tokenizer(nb_words=MAX_NB_WORDS, split=' ')
tokenizer.fit_on_texts(data['content'].values)
X = tokenizer.texts_to_sequences(data['content'].values)
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)

In [ ]:
data['content']

**The giving dataset contains lots of garbage words (*as obvious because people use slang and shortform at twitter due to 140 chars limitation*) as you see above, many word like layin,ughhhhwaitin,lovelylisaj,sendsome2me, n, niariley, mopedronin etc have no meaning which makes data noisy very noisy which ultimately give poor output**.

In [ ]:
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Y = data['sentiment']
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(Y)
Y=le.transform(Y) 
labels = to_categorical(np.asarray(Y))
print('Shape of data tensor:', X.shape)
print('Shape of label tensor:', labels.shape)

VALIDATION_SPLIT=0.20

# split the data into a training set and a validation set
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
labels = labels[indices]
nb_validation_samples = int(VALIDATION_SPLIT * X.shape[0])

x_train = X[:-nb_validation_samples]
y_train = labels[:-nb_validation_samples]
x_val = X[-nb_validation_samples:]
y_val = labels[-nb_validation_samples:]

In [ ]:
embeddings_index = {}
f = open('glove.42B.300d.txt', encoding="utf8")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

In [ ]:
EMBEDDING_DIM=300

In [ ]:
embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # words not found in embedding index will be all-zeros.
        embedding_matrix[i] = embedding_vector

In [ ]:
from keras.layers import Embedding

embedding_layer = Embedding(len(word_index) + 1,
                            EMBEDDING_DIM,
                            weights=[embedding_matrix],
                            input_length=MAX_SEQUENCE_LENGTH,
                            trainable=False)

In [ ]:
embed_dim = 300
lstm_out = 196

model = Sequential()
model.add(embedding_layer)
model.add(LSTM(lstm_out, dropout_U=0.2, dropout_W=0.25))
model.add(Dense(9,activation='softmax'))
model.compile(loss = 'categorical_crossentropy', optimizer='adam',metrics = ['accuracy'])
print(model.summary())

Hereby I declare the train and test dataset.

In [ ]:
# Y = pd.get_dummies(data['sentiment']).values
# X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.33, random_state = 42)
# print(X_train.shape,Y_train.shape)
# print(X_test.shape,Y_test.shape)

Here we train the Network. We should run much more than 7 epoch, but I would have to wait forever for kaggle, so it is 7 for now.

In [ ]:
model.fit(x_train, y_train,
          batch_size=128,
          epochs=10,
          validation_data=(x_val, y_val))

Extracting a validation set, and measuring score and accuracy.

In [ ]:
score,acc = model.evaluate(X_test, Y_test, verbose = 2, batch_size = batch_size)
print("score: %.2f" % (score))
print("acc: %.2f" % (acc))

Finally measuring the number of correct guesses.  It is clear that finding negative tweets goes very well for the Network but deciding whether is positive is not really. My educated guess here is that the positive training set is dramatically smaller than the negative, hence the "bad" results for positive tweets.

In [ ]:
import xgboost as xgb
gbm = xgb.XGBClassifier(max_depth=3, n_estimators=300, learning_rate=0.05).fit(inp, target)
predictions = gbm.predict(X_test)
from sklearn import cross_validation
acc_scores = cross_validation.cross_val_score(gbm,inp, target, cv=5)
print (acc_scores)
print(metrics.accuracy_score(Y_test, predictions))

In [ ]:
from sklearn import linear_model
logisticReg = linear_model.LogisticRegression(multi_class='multinomial', solver='lbfgs')
logisticReg.fit(inp, target)
predictions = logisticReg.predict(X_test)
print(metrics.accuracy_score(Y_test, predictions))

In [ ]:
Y = pd.get_dummies(data['sentiment'])
print(Y.values)

In [ ]:
Y = data['sentiment']
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(Y)
Y=le.transform(Y) 
inp, X_test, target, Y_test = train_test_split(X,Y, test_size = 0.2, random_state = 42)

In [ ]:
Y